# Ejemplo rápido de uso con EEG

Este notebook muestra el flujo mínimo para cargar un EEG, calcular conectividad y visualizar el primer grafo generado por la librería.

## 1. Importaciones

Importamos la API pública de EEGraph y unas utilidades para manejar rutas y matrices.

In [ ]:
from pathlib import Path
import numpy as np
from eegraph.graph import Graph

## 2. Ruta del EEG de entrada

Sustituye esta ruta por tu archivo `.edf` real. Si trabajas con otro formato soportado por EEGraph, cambia la extensión y mantén la misma llamada a `load_data()`.

In [ ]:
RUTA_EEG = Path("./datos/mi_eeg.edf")
SALIDA = Path("./salidas")
SALIDA.mkdir(exist_ok=True)

## 3. Carga de los datos

En EEG basta con indicar la ruta y la modalidad `eeg`.

In [ ]:
G = Graph()
G.load_data(
    path=str(RUTA_EEG),
    modality="eeg"
)

print("Modalidad:", G.modality)
print("Canales cargados:", G.ch_names)

## 4. Cálculo de conectividad

Aquí usamos `pearson_correlation` con ventanas de 2 segundos. En EEG, la salida puede contener varios grafos, por ejemplo uno por ventana temporal.

In [ ]:
graphs, connectivity_matrix = G.modelate(
    window_size=2,
    connectivity="pearson_correlation"
)

print("Tipo de salida de graphs:", type(graphs))
print("Shape de la matriz de conectividad:", np.array(connectivity_matrix).shape)

## 5. Extraer un grafo real para visualizarlo

La salida de EEG puede venir en forma de grafo directo, lista, tupla o diccionario. Esta función busca recursivamente el primer grafo disponible.

In [ ]:
def extraer_primer_grafo(objeto):
    if hasattr(objeto, "is_directed") and callable(objeto.is_directed):
        return objeto
    if isinstance(objeto, dict):
        for _, valor in objeto.items():
            encontrado = extraer_primer_grafo(valor)
            if encontrado is not None:
                return encontrado
    if isinstance(objeto, (list, tuple)):
        for elemento in objeto:
            encontrado = extraer_primer_grafo(elemento)
            if encontrado is not None:
                return encontrado
    return None

grafo_eeg = extraer_primer_grafo(graphs)
print("Número de nodos:", grafo_eeg.number_of_nodes())
print("Número de aristas:", grafo_eeg.number_of_edges())
print("Primeros nodos:", list(grafo_eeg.nodes())[:10])

## 6. Visualización

Generamos un HTML interactivo con Plotly. Si no quieres que el navegador se abra automáticamente, usa `auto_open=False`.

In [ ]:
G.visualize_html(
    grafo_eeg,
    SALIDA / "eeg_ejemplo",
    auto_open=False
)

print("Archivo generado:", SALIDA / "eeg_ejemplo_plot.html")

## 7. Qué te llevas de este ejemplo

Con EEG, el flujo básico es:

1. crear `Graph()`
2. llamar a `load_data(..., modality="eeg")`
3. llamar a `modelate(...)`
4. extraer un grafo de la salida
5. visualizarlo con `visualize_html()` o `visualize_png()`